# 📘 Bài 4 — Eval: đo chất lượng LLM

**Tuần 19** — bài quan trọng nhất của Phase 07.

> **Vấn đề:** bạn sửa prompt, chạy thử 2–3 lần, thấy "có vẻ tốt hơn", rồi deploy.
> Hai tuần sau khách báo lỗi ở một trường hợp mà prompt cũ xử lý đúng.
>
> **Bạn không có cách nào biết mình đã làm hỏng cái gì.**

Eval là bộ test tự động cho hệ thống LLM. Không có nó, bạn đang lập trình mù.

In [ ]:
import os
import json
from pathlib import Path

# Nap .env neu co
try:
    from dotenv import load_dotenv
    for p in (Path(".env"), Path("../../../.env")):
        if p.exists():
            load_dotenv(p)
            break
except ImportError:
    pass

import anthropic

MODEL_RE = "claude-haiku-4-5"        # $1 vào / $5 ra  — mặc định của khoá học
MODEL_MANH = "claude-opus-5"         # $5 vào / $25 ra — chỉ dùng khi so sánh

CO_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
client = anthropic.Anthropic() if CO_KEY else None


def goi(**kwargs):
    """Gọi API nếu có key; nếu chưa có thì in thông báo và trả về None.

    Nhờ hàm này, notebook chạy được cả khi bạn chưa lấy API key —
    bạn vẫn đọc được code và giải thích.
    """
    if not CO_KEY:
        print("⏭  Bỏ qua — chưa có ANTHROPIC_API_KEY (xem SETUP.md mục 5)")
        return None
    return client.messages.create(**kwargs)


def van_ban(response) -> str:
    """Lấy toàn bộ text từ response. KHÔNG dùng response.content[0].text!"""
    if response is None:
        return ""
    return "".join(b.text for b in response.content if b.type == "text")


print("anthropic SDK:", anthropic.__version__)
print("API key      :", "đã có ✓" if CO_KEY else "CHƯA CÓ — notebook vẫn chạy được")

## 1. Vì sao "chạy thử vài lần" không đủ

In [ ]:
# Cùng một prompt, cùng một model, chạy 5 lần
if CO_KEY:
    prompt = "Review: 'Giao nhanh nhưng đóng gói hơi ẩu.' Phân loại: tich_cuc/tieu_cuc/trung_tinh. Chỉ trả về một từ."
    kq = []
    for _ in range(5):
        r = client.messages.create(model=MODEL_RE, max_tokens=20, temperature=1.0,
                                   messages=[{"role": "user", "content": prompt}])
        kq.append(van_ban(r).strip())
    print(f"  5 lần chạy: {kq}")
    print(f"  Số kết quả khác nhau: {len(set(kq))}")
else:
    print("⏭  Bỏ qua — chưa có API key")
    print("  Kết quả điển hình: ['trung_tinh', 'trung_tinh', 'tich_cuc', 'trung_tinh', 'tich_cuc']")

print("""
  Cùng đầu vào -> có thể ra kết quả khác nhau.
  => Chạy thử 1 lần KHÔNG chứng minh được gì.
  => Trong eval, LUÔN dùng temperature=0 để kết quả tái lập được.""")

**Ba lý do bắt buộc phải có eval:**

| Lý do | Nếu không có eval |
|---|---|
| **Đầu ra không tất định** | Bạn nhầm may mắn với chất lượng |
| **Không có "đúng/sai" tuyệt đối** | Bạn tranh cãi cảm tính thay vì nhìn số |
| **Sửa chỗ này hỏng chỗ kia** | Bạn không phát hiện được **thoái lui (regression)** |

> 🔗 Bạn đã gặp ý này ở Phase 04: **train/test split** tồn tại vì đo trên dữ liệu đã thấy là tự lừa mình. Eval là cùng một nguyên tắc, áp cho LLM.

## 2. Bộ test — trái tim của eval

In [ ]:
BO_TEST = [
    # Trường hợp dễ — phải luôn đúng
    {"ten": "ro_rang_tich_cuc", "dau_vao": "Sản phẩm tuyệt vời, rất hài lòng!",
     "mong_doi": "tich_cuc"},
    {"ten": "ro_rang_tieu_cuc", "dau_vao": "Hàng lỗi, dịch vụ tệ hại.",
     "mong_doi": "tieu_cuc"},

    # Trường hợp khó — đây mới là chỗ eval có giá trị
    {"ten": "lan_lon", "dau_vao": "Đồ ăn ngon nhưng giao muộn 1 tiếng.",
     "mong_doi": "trung_tinh"},
    {"ten": "mia_mai", "dau_vao": "Tuyệt vời, đợi 3 tuần mới nhận được hàng. Cảm ơn nhé.",
     "mong_doi": "tieu_cuc"},
    {"ten": "phu_dinh_kep", "dau_vao": "Không thể nói là không hài lòng.",
     "mong_doi": "tich_cuc"},

    # Trường hợp biên — nơi hệ thống hay sập
    {"ten": "rat_ngan", "dau_vao": "ok", "mong_doi": "trung_tinh"},
    {"ten": "chi_emoji", "dau_vao": "😡😡😡", "mong_doi": "tieu_cuc"},
    {"ten": "khong_lien_quan", "dau_vao": "Hôm nay trời mưa.", "mong_doi": "trung_tinh"},
]

print(f"  {len(BO_TEST)} case: 2 dễ, 3 khó, 3 biên")

> ⚠️ **Bộ test toàn case dễ là bộ test vô dụng.** Nó sẽ luôn cho 100% và không bao giờ phát hiện được gì. Giá trị của eval nằm ở **case khó và case biên**.
>
> **Nguồn case tốt nhất là lỗi thật.** Mỗi lần người dùng báo một kết quả sai, biến nó thành một case. Bộ test của bạn lớn dần theo đúng những chỗ hệ thống hay sai.

**Bao nhiêu case là đủ?**

| Số case | Dùng để |
|---|---|
| 10–20 | Lặp nhanh khi đang sửa prompt |
| 50–100 | Quyết định deploy |
| 200+ | So sánh model, thay đổi kiến trúc |

Bắt đầu từ 10. **Một bộ eval 10 case đang chạy tốt hơn một bộ 200 case bạn chưa viết.**

## 3. Hàm chấm điểm — từ dễ đến khó

In [ ]:
def khop_chinh_xac(dau_ra, mong_doi):
    return dau_ra.strip().lower() == mong_doi.strip().lower()


def chua_tat_ca(dau_ra, tu_khoa):
    t = dau_ra.lower()
    return all(k.lower() in t for k in tu_khoa)


def json_hop_le(dau_ra, khoa_bat_buoc=None):
    try:
        d = json.loads(dau_ra)
    except json.JSONDecodeError:
        return False
    if khoa_bat_buoc is None:
        return True
    return isinstance(d, dict) and all(k in d for k in khoa_bat_buoc)


print(f"  khop_chinh_xac : {khop_chinh_xac('  Tich_Cuc ', 'tich_cuc')}")
print(f"  chua_tat_ca    : {chua_tat_ca('Giá 100k, giao 2 ngày', ['100k', '2 ngày'])}")
print(f"  json_hop_le    : {json_hop_le(chr(123)+chr(34)+'a'+chr(34)+': 1'+chr(125), ['a'])}")

**Ba tầng hàm chấm — chọn tầng THẤP NHẤT còn dùng được:**

| Tầng | Cách chấm | Chi phí | Khi nào dùng |
|---|---|---|---|
| **1. Xác định** | So chuỗi, regex, parse JSON | Miễn phí, tức thì | Phân loại, trích xuất, định dạng |
| **2. Có tham chiếu** | So với đáp án chuẩn theo từng trường | Miễn phí | Trích xuất có nhiều trường |
| **3. LLM-as-judge** | Một model khác chấm | Tốn tiền, chậm, có nhiễu | Văn xuôi tự do, giọng điệu |

> 🎯 **Đừng nhảy lên tầng 3.** Người mới hay nghĩ "đầu ra là văn bản tự do nên phải dùng judge". Thực tế: nếu bạn ép model trả **structured output** (Tuần 17), phần lớn bài toán tụt xuống tầng 1 — miễn phí, tức thì, không nhiễu.
>
> **Thiết kế đầu ra để dễ chấm** là một quyết định kỹ thuật, không phải ăn may.

## 4. ⭐ Khung eval

In [ ]:
def chay_eval(bo_test, goi_model, cham):
    """Chạy eval. `goi_model` là một HÀM — nhờ vậy test được offline.

    Đây là dependency injection: khung eval không biết gì về API.
    Bạn truyền model thật, model giả, hay baseline vào đều được.
    """
    ket_qua = []
    for i, case in enumerate(bo_test):
        ten = case.get("ten", f"case_{i}")
        try:
            dau_ra = goi_model(case["dau_vao"])
            diem = float(cham(dau_ra, case["mong_doi"]))
            loi = None
        except Exception as e:
            dau_ra, diem, loi = None, 0.0, f"{type(e).__name__}: {e}"
        ket_qua.append({"ten": ten, "dau_vao": case["dau_vao"],
                        "dau_ra": dau_ra, "diem": diem, "loi": loi})
    return ket_qua


def tom_tat(ket_qua):
    n = len(ket_qua)
    if n == 0:
        return {"so_case": 0, "diem_tb": 0.0, "so_dat": 0, "ty_le_dat": 0.0, "so_loi": 0}
    dat = sum(1 for r in ket_qua if r["diem"] >= 1.0)
    return {
        "so_case": n,
        "diem_tb": sum(r["diem"] for r in ket_qua) / n,
        "so_dat": dat,
        "ty_le_dat": dat / n,
        "so_loi": sum(1 for r in ket_qua if r["loi"] is not None),
    }


print("Khung eval sẵn sàng — chưa cần API key nào.")

> 🔑 **`goi_model` là tham số, không phải lời gọi API cứng.** Đây là lý do bài tập `ex03_eval.py` chạy và chấm điểm được **không cần API key**, và cũng là lý do bạn có thể so sánh model thật với **baseline** ở mục 6.
>
> Cùng một ý bạn đã thấy ở Phase 04 với `Pipeline`: tách *cái gì được làm* khỏi *ai làm nó*.

## 5. Chạy eval thật

In [ ]:
PROMPT_V1 = """Phân loại cảm xúc của review sau.

<review>
{review}
</review>

Chỉ trả về đúng một từ: tich_cuc, tieu_cuc, hoặc trung_tinh."""


def tao_model(prompt_mau):
    """Trả về một HÀM gọi model với prompt cho trước."""
    def f(dau_vao):
        r = client.messages.create(
            model=MODEL_RE, max_tokens=20, temperature=0,      # temperature=0!
            messages=[{"role": "user", "content": prompt_mau.format(review=dau_vao)}],
        )
        return van_ban(r).strip()
    return f


def in_bang(ket_qua, tieu_de):
    print(f"\n  === {tieu_de} ===")
    print(f"  {'case':<20} {'điểm':>5}  đầu ra")
    print("  " + "-" * 55)
    for r in ket_qua:
        dau = "✅" if r["diem"] >= 1.0 else "❌"
        print(f"  {r['ten']:<20} {dau:>4}  {str(r['dau_ra'])[:28]!r}")
    t = tom_tat(ket_qua)
    print(f"  -> {t['so_dat']}/{t['so_case']} đạt ({t['ty_le_dat']:.0%}), {t['so_loi']} lỗi")


if CO_KEY:
    kq_v1 = chay_eval(BO_TEST, tao_model(PROMPT_V1), khop_chinh_xac)
    in_bang(kq_v1, "Prompt v1")
else:
    print("⏭  Bỏ qua — chưa có API key")
    print("""
  Kết quả điển hình của v1: 5/8 đạt (63%)
     ❌ mia_mai        -> 'tich_cuc'    (không nhận ra mỉa mai)
     ❌ phu_dinh_kep   -> 'trung_tinh'  (không xử lý phủ định kép)
     ❌ chi_emoji      -> 'trung_tinh'  (không hiểu emoji)""")

## 6. ⭐ Baseline — con số 63% có tốt không?

In [ ]:
def baseline_hang_so(gia_tri):
    """Model ngu nhất có thể: luôn trả về cùng một nhãn."""
    return lambda _: gia_tri


for nhan in ["tich_cuc", "tieu_cuc", "trung_tinh"]:
    t = tom_tat(chay_eval(BO_TEST, baseline_hang_so(nhan), khop_chinh_xac))
    print(f"  Luôn đoán {nhan:<12} -> {t['so_dat']}/{t['so_case']} ({t['ty_le_dat']:.0%})")

print("""
  Baseline tốt nhất đạt 38% mà KHÔNG đọc gì cả.
  => 63% của model chỉ hơn baseline 25 điểm.
  => Nếu bộ test của bạn có 90% case là 'trung_tinh', một model
     luôn đoán 'trung_tinh' sẽ đạt 90% — và hoàn toàn vô dụng.""")

> 🔗 **Bạn đã gặp cái bẫy này ở Phase 04** với dữ liệu churn 26%: một model luôn đoán "không rời bỏ" đạt 74% accuracy. Con số cao mà vô dụng.
>
> **Luôn báo cáo eval kèm baseline.** "Model đạt 85%" là một câu vô nghĩa cho đến khi bạn nói baseline là bao nhiêu.

## 7. Cải tiến prompt và ĐO lại

In [ ]:
PROMPT_V2 = """Phân loại cảm xúc của review khách hàng.

Quy tắc:
- tich_cuc: khách hài lòng tổng thể (kể cả khi diễn đạt qua phủ định kép,
  ví dụ "không thể nói là không hài lòng")
- tieu_cuc: khách không hài lòng, PHÀN NÀN, hoặc MỈA MAI
  (khen ngợi kèm chi tiết tiêu cực rõ ràng là mỉa mai)
- trung_tinh: khen chê cân bằng, nội dung quá ngắn/mơ hồ, hoặc không liên quan

Chú ý: emoji cũng mang cảm xúc (😡 = tiêu cực, 😍 = tích cực).

<review>
{review}
</review>

Chỉ trả về đúng một từ: tich_cuc, tieu_cuc, hoặc trung_tinh."""

if CO_KEY:
    kq_v2 = chay_eval(BO_TEST, tao_model(PROMPT_V2), khop_chinh_xac)
    in_bang(kq_v2, "Prompt v2")
else:
    print("⏭  Bỏ qua — chưa có API key")
    print("  Kết quả điển hình của v2: 7/8 đạt (88%)")

## 8. ⭐⭐ So sánh phiên bản — phát hiện THOÁI LUI

In [ ]:
def so_sanh_phien_ban(cu, moi):
    """Tìm case ĐANG ĐÚNG mà phiên bản mới làm HỎNG."""
    d_cu = {r["ten"]: r["diem"] for r in cu}
    d_moi = {r["ten"]: r["diem"] for r in moi}
    chung = sorted(set(d_cu) & set(d_moi))

    thoai_lui = [t for t in chung if d_moi[t] < d_cu[t]]
    cai_thien = [t for t in chung if d_moi[t] > d_cu[t]]
    return {
        "diem_tb_cu": sum(d_cu[t] for t in chung) / len(chung) if chung else 0.0,
        "diem_tb_moi": sum(d_moi[t] for t in chung) / len(chung) if chung else 0.0,
        "thoai_lui": thoai_lui,
        "cai_thien": cai_thien,
        "khong_doi": len(chung) - len(thoai_lui) - len(cai_thien),
    }


# Minh hoạ bằng dữ liệu cố định — thấy rõ vấn đề, không cần API
cu = [{"ten": "ro_rang_tich_cuc", "diem": 1.0}, {"ten": "ro_rang_tieu_cuc", "diem": 1.0},
      {"ten": "lan_lon", "diem": 1.0}, {"ten": "mia_mai", "diem": 0.0},
      {"ten": "phu_dinh_kep", "diem": 0.0}, {"ten": "rat_ngan", "diem": 1.0},
      {"ten": "chi_emoji", "diem": 0.0}, {"ten": "khong_lien_quan", "diem": 1.0}]

moi = [{"ten": "ro_rang_tich_cuc", "diem": 1.0}, {"ten": "ro_rang_tieu_cuc", "diem": 1.0},
       {"ten": "lan_lon", "diem": 0.0},          # ← HỎNG!
       {"ten": "mia_mai", "diem": 1.0}, {"ten": "phu_dinh_kep", "diem": 1.0},
       {"ten": "rat_ngan", "diem": 1.0}, {"ten": "chi_emoji", "diem": 1.0},
       {"ten": "khong_lien_quan", "diem": 1.0}]

ss = so_sanh_phien_ban(cu, moi)
print(f"  Điểm TB: {ss['diem_tb_cu']:.0%} -> {ss['diem_tb_moi']:.0%}   (tăng 25 điểm 🎉)")
print(f"  Cải thiện : {ss['cai_thien']}")
print(f"  ⚠️  THOÁI LUI: {ss['thoai_lui']}")
print("""
  Điểm trung bình TĂNG, nhưng 'lan_lon' đã HỎNG.
  Nếu chỉ nhìn con số tổng, bạn deploy và không hề biết.

  Prompt v2 dạy model nhận mỉa mai -> nó thành ra quá nhạy,
  gán 'tieu_cuc' cho cả review khen-chê cân bằng.

  Đây là lý do so sánh phải theo TỪNG CASE, không phải theo trung bình.""")

> 🎯 **Đây là bài học lớn nhất của Phase 07.**
>
> Một con số tổng có thể **cải thiện trong khi hệ thống đang xấu đi** ở đúng chỗ bạn quan tâm. Điều tương tự xảy ra với accuracy trong ML, với độ phủ test trong phần mềm, và với mọi chỉ số gộp.
>
> Quy trình đúng khi sửa prompt:
>
> ```
> ① Chạy eval trên prompt hiện tại   -> lưu kết quả v1
> ② Sửa prompt
> ③ Chạy eval lại                    -> kết quả v2
> ④ so_sanh_phien_ban(v1, v2)
> ⑤ Nếu thoai_lui KHÔNG rỗng -> ĐỌC từng case hỏng trước khi deploy
> ```

## 9. LLM-as-judge — khi không có đáp án chuẩn

In [ ]:
MAU_JUDGE = """Bạn là giám khảo đánh giá câu trả lời của trợ lý AI.

<cau_hoi>
{cau_hoi}
</cau_hoi>

<dap_an_chuan>
{dap_an}
</dap_an_chuan>

<cau_tra_loi>
{tra_loi}
</cau_tra_loi>

Chấm điểm từ 1 đến 5:
5 = chính xác và đầy đủ như đáp án chuẩn
4 = chính xác, thiếu chi tiết nhỏ
3 = đúng một phần, thiếu ý quan trọng
2 = phần lớn sai
1 = hoàn toàn sai hoặc lạc đề

Chỉ trả về JSON: {{"diem": <1-5>, "ly_do": "<một câu>"}}"""


def parse_judge(van_ban_judge):
    """Judge cũng có thể trả về rác — phải xử lý."""
    try:
        i, j = van_ban_judge.index("{"), van_ban_judge.rindex("}")
        d = json.loads(van_ban_judge[i:j + 1])
        diem = int(d["diem"])
        if not 1 <= diem <= 5:
            return 0.0, "Điểm ngoài khoảng 1-5"
        return (diem - 1) / 4, d.get("ly_do", "")      # chuẩn hoá về 0.0-1.0
    except (ValueError, KeyError, TypeError, json.JSONDecodeError):
        return 0.0, "Không parse được kết quả judge"


for mau in ['{"diem": 5, "ly_do": "Chính xác"}', '{"diem": 3, "ly_do": "Thiếu ý"}',
            'Đánh giá: {"diem": 4, "ly_do": "Tốt"}', 'xin lỗi tôi không chấm được']:
    print(f"  {mau[:42]:<44} -> {parse_judge(mau)}")

**Bốn quy tắc dùng judge:**

| # | Quy tắc | Vì sao |
|---|---|---|
| 1 | Thang điểm **rời rạc** (1–5), không phải 0–100 | Model không phân biệt được 73 với 76 |
| 2 | **Mô tả rõ từng mức** | "5 = chính xác và đầy đủ", không phải "5 = tốt" |
| 3 | Bắt trả **JSON có `ly_do`** | Không có lý do thì bạn không debug được |
| 4 | **Kiểm định judge** trên ~30 case bạn đã tự chấm tay | Judge cũng sai — bạn phải biết nó sai bao nhiêu |

> ⚠️ **Judge không phải sự thật.** Nó là một model, với cùng những điểm yếu bạn đang cố đo. Nếu bạn chưa từng đối chiếu judge với đánh giá của chính mình, bạn không biết mình đang đo cái gì.
>
> 💰 Judge tốn tiền: mỗi case là **hai** lời gọi API. Với bộ test 100 case, đó là 200 lượt gọi mỗi lần chạy eval.

## 10. Chi phí eval

In [ ]:
def chi_phi_eval(so_case, tok_vao, tok_ra, gia_vao, gia_ra, co_judge=False):
    c = so_case * (tok_vao / 1e6 * gia_vao + tok_ra / 1e6 * gia_ra)
    if co_judge:
        c += so_case * ((tok_vao + tok_ra + 200) / 1e6 * gia_vao + 100 / 1e6 * gia_ra)
    return c


print(f"  {'kịch bản':<40} {'chi phí/lần chạy':>17}")
print("  " + "-" * 58)
for ten, n, judge in [
    ("10 case, chấm xác định (lặp nhanh)", 10, False),
    ("100 case, chấm xác định (deploy)", 100, False),
    ("100 case, có LLM judge", 100, True),
    ("1000 case, có LLM judge", 1000, True),
]:
    c = chi_phi_eval(n, 400, 50, 1.00, 5.00, judge)
    print(f"  {ten:<40} {'$' + format(c, '.4f'):>17}")

print("""
  Eval xác định gần như MIỄN PHÍ. Chạy nó mỗi lần sửa prompt.
  Eval có judge thì đắt -> chạy trước khi deploy, không phải mỗi lần sửa.

  Nếu tổng eval của cả khoá học này < $1, đó là khoản đầu tư
  rẻ nhất bạn từng bỏ ra để không deploy một hệ thống hỏng.""")

## 11. Tổng kết Phase 07

Bạn đã đi từ "gọi được API" đến "chứng minh được hệ thống của mình hoạt động":

| Tuần | Kỹ năng | Câu hỏi nó trả lời |
|---|---|---|
| 16 | API, token, chi phí, `stop_reason` | *Gọi thế nào và tốn bao nhiêu?* |
| 17 | Prompt, structured output | *Làm sao lấy được dữ liệu code dùng được?* |
| 18 | Tool use, streaming, caching | *Làm sao thành một sản phẩm thật?* |
| 19 | **Eval** | *Làm sao biết nó thực sự hoạt động?* |

> 🎯 **Điều phân biệt AI Engineer với người biết gọi API:** người biết gọi API sửa prompt cho đến khi "thấy ổn". AI Engineer có bộ eval, biết baseline, và phát hiện được thoái lui trước khi khách hàng phát hiện.

## ✍️ Bài tập & Project

```bash
pytest tests/phase07/ -v          # cả 3 bài, không cần API key
```

- **Mini-project:** `curriculum/07-llm-engineering/mini_project/README.md`
- **Project lớn:** `projects/P5-structured-extractor/README.md`
- **Ôn phỏng vấn:** `resources/interview/phase07-llm.md`

**Tiếp theo — Phase 08 (RAG):** eval bạn vừa học sẽ được dùng ngay để đo chất lượng retrieval. Không có eval, bạn sẽ không phân biệt được "retrieval kém" với "prompt kém".